# 两个 DP-FedAvg sweep 的 reconciliation

对 GEF2017 和 AusGrid2013 各 12 组已完成的 20-round sweep，计算 **base、mints、wlsv、ols** 的预测与指标。

- 直接复用 `shared_modules/FHF_Cloud.py` 的原有实现：mints = `mint_shrinkage`，wlsv = `mint_var`，ols = `mint_ols`，`ridge=1e-6`。
- 每组使用该组保存的**训练残差**估计矩阵；测试真值仅用于评分。无需重训，也不另外添加预测噪声或残差噪声。
- MASE / RMSSE 采用与原 sweep 一致的训练目标 lag-1 差分分母；先按序列计算，再对全部序列取算术平均。
- 每次运行重新计算，验证 base 指标与原 sweep 一致、`P S ≈ S`、`P² ≈ P` 以及校正预测的层级加总一致性。
- 所有结果保存至**`DP_fhf/outputs/dp_sweep_reconciliation/`**，输入 sweep 保持不变。

`training_epsilon` / `training_delta` 仅记录原 DP-FedAvg 的训练隐私参数；本次训练残差协方差估计不附带新的 DP 保证。

目录调整：已有计算结果已整体迁入上述目录，以下结果表保留；日志中的文件位置已更新为当前位置。以后运行也写入此目录。

In [ ]:
# Locate the project package from a notebook folder or the project root.
from pathlib import Path
import os
import sys
_pphf_start = Path.cwd().resolve()
_pphf_override = os.environ.get("PPHF_PROJECT_ROOT") or os.environ.get("DP_FHF_PROJECT_ROOT")
_pphf_candidates = ([Path(_pphf_override).expanduser()] if _pphf_override else []) + [_pphf_start, *_pphf_start.parents]
PROJECT_ROOT = next((p.resolve() for p in _pphf_candidates if (p / "PPHF_modules" / "fhf" / "FHF_runner.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate PPHF_modules. Set PPHF_PROJECT_ROOT or start Jupyter inside the project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pathlib import Path
import os
import sys
import importlib
import pandas as pd
from IPython.display import display

# 在项目任意子目录启动；也可通过环境变量指定项目根目录。
candidates = []
if os.environ.get("DP_FHF_PROJECT_ROOT"):
    candidates.append(Path(os.environ["DP_FHF_PROJECT_ROOT"]).expanduser())
candidates.extend([Path.cwd(), *Path.cwd().parents])
ROOT = next((p.resolve() for p in candidates
             if (p / "PPHF_modules" / "fhf").is_dir() and (p / "DP_fhf").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Set DP_FHF_PROJECT_ROOT to the FHF_NDSH project directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from PPHF_modules.dp_fhf import sweep_reconciliation
importlib.reload(sweep_reconciliation)

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda value: f"{value:.6f}")
print("Project:", ROOT)
print("Python:", sys.executable)

In [ ]:
DATASETS = ("GEF2017", "AusGrid2013")
OUTPUT_ROOT = ROOT / "DP_fhf" / "outputs" / "dp_sweep_reconciliation"
RIDGE = 1e-6
SAVE_FORECASTS = True  # 保存逐点预测 CSV.gz，可直接用 pandas.read_csv 读取。
RUN_IDS = None         # None = 全部 24 组；可改成 ["nm_0p3__clip_1"] 做子集检查。

print("Output:", OUTPUT_ROOT)
print("Methods:", sweep_reconciliation.METHODS)
print("Matrix source: each run's saved training residuals")

## 计算并保存

较大的 AusGrid 输出需要一些时间压缩写入。每完成一组，显示四种方法的 MASE。
已有本 notebook 生成的同名结果会重新计算并替换；该组全部验证和写入成功后才发布结果。

In [ ]:
summary = sweep_reconciliation.run_sweep_reconciliation(
    root=ROOT,
    datasets=DATASETS,
    output_root=OUTPUT_ROOT,
    ridge=RIDGE,
    save_forecasts=SAVE_FORECASTS,
    run_ids=RUN_IDS,
)
assert len(summary) == 4 * summary[["dataset", "run_id"]].drop_duplicates().shape[0]
print("Completed method rows:", len(summary))

## 每组参数的 MASE / RMSSE

按 clipping 与 noise 排序，便于比较噪声变化。base 是该组原始预测。

In [ ]:
for dataset in DATASETS:
    print(f"\n{dataset}")
    table = summary.loc[summary.dataset == dataset].pivot(
        index=["clip_norm", "noise_multiplier"], columns="method", values=["MASE", "RMSSE"]
    ).sort_index()
    table = table.reindex(columns=pd.MultiIndex.from_product(
        [["MASE", "RMSSE"], ["base", "mints", "wlsv", "ols"]], names=table.columns.names))
    display(table)

## 相对于同组 base 的变化

负数表示误差降低。每个数据集 / 方法各有 12 个参数组合；以下仅作描述性比较，不是多 seed 的统计结论。

In [ ]:
comparison_columns = ["dataset", "run_id", "method", "MASE", "RMSSE",
                      "change_MASE_pct_vs_base", "change_RMSSE_pct_vs_base"]
display(summary.loc[summary.method != "base", comparison_columns])
best = summary.loc[summary.groupby(["dataset", "method"])["MASE"].idxmin(),
                   ["dataset", "method", "run_id", "clip_norm", "noise_multiplier", "MASE", "RMSSE"]]
print("Lowest mean MASE within each dataset / method:")
display(best.sort_values(["dataset", "method"]))

## 验证与文件位置

- `metrics_summary.csv`：全部 96 行（24 组 × base + 3 种 reconciliation），包含四项指标及相对 base 的变化。
- `metrics_wide.csv`：24 行，四种方法的指标并排。
- `per_node_metrics.csv`：全部逐序列指标。
- `matrix_diagnostics.csv`：层级一致性与矩阵验证。
- `<dataset>/<run_id>/`：该组指标、`reconciliation_matrices.npz`（S / G / P / W）、`reconciled_forecasts.csv.gz` 和输入文件 SHA256 / 计算设置。
- `manifest.json`：全批次完成记录。若执行中断，不会保留旧的全批次成功标记。

In [ ]:
import json
manifest = json.loads((OUTPUT_ROOT / "manifest.json").read_text())
checks = pd.read_csv(OUTPUT_ROOT / "matrix_diagnostics.csv")
assert manifest["status"] == "complete"
assert checks["PS_minus_S_max_abs"].max() < 1e-7
assert checks["coherence_relative"].max() < 1e-9
print("Runs:", manifest["runs_completed"])
print("Maximum |PS-S|:", checks["PS_minus_S_max_abs"].max())
print("Maximum relative coherence error:", checks["coherence_relative"].max())
print("Summary:", OUTPUT_ROOT / "metrics_summary.csv")
print("Wide comparison:", OUTPUT_ROOT / "metrics_wide.csv")
# 示例：pd.read_csv(OUTPUT_ROOT / "GEF2017" / "nm_0p3__clip_1" / "reconciled_forecasts.csv.gz")